# 07. Evaluation, Error Analysis, and Interpretability

## Which Model Should We Trust as Our Final Submission?

At this point in the project, we have moved from understanding the data to building baselines, RoBERTa models, imbalance-aware variants, safe data strategies, and an ensemble. The question is no longer only “which model has the highest number?” but **which model should we trust as our final submission and how do we explain its behavior?**

This notebook compares all completed model versions on the shared validation split. It also studies per-class behavior, confusion patterns, confidence, and concrete clinical-literal examples.


## Metrics We Use

- **Accuracy**: the competition-oriented metric. It counts exact matches of the single `y_category` prefix.
- **Macro precision / recall / F1**: every category contributes equally, so this is important for imbalance and rare categories.
- **Weighted F1**: averages F1 by class support, so it sits between accuracy and macro F1 in spirit.
- **Per-class recall**: tells us which ICD prefixes the model fails to recover.
- **Top-k accuracy**: available for models with probabilities; it tells us whether the correct category appears among the model's alternatives.
- **Confidence and confidence margin**: useful for error analysis, not as a guarantee of correctness.

Accuracy matters because it is the leaderboard metric. Macro F1 matters because ICD categories are imbalanced: a model can look good by accuracy while still failing rare categories.


In [ ]:
import pandas as pd
from pathlib import Path

comparison = pd.read_csv('../reports/tables/final_experiment_comparison.csv')
per_class = pd.read_csv('../reports/tables/final_per_class_metrics.csv')
errors = pd.read_csv('../reports/tables/final_error_examples.csv')
confusions = pd.read_csv('../reports/tables/final_top_confusions.csv')

comparison[['model_id', 'family', 'accuracy', 'macro_precision', 'macro_recall', 'macro_f1', 'weighted_f1', 'top_3_accuracy']]


## What We Found

After the v10 search, `v10_vote_diverse_no_retrieval` is the final public-leaderboard candidate and also improves over the earlier `v09_ensemble` on validation accuracy:

| model | accuracy | macro F1 | weighted F1 |
|---|---:|---:|---:|
| best single model, `v04_roberta_cls` | 0.5693 | 0.4943 | 0.5543 |
| earlier ensemble, `v09_ensemble` | 0.5766 | 0.5063 | 0.5615 |
| final public candidate, `v10_vote_diverse_no_retrieval` | 0.5796 | 0.4967 | 0.5613 |

The result has a trade-off: `v10` improves validation accuracy and has the best verified Kaggle public score, while `v09` keeps a slightly stronger macro F1. Because the competition is accuracy-oriented and the public leaderboard confirms `v10`, we select `v10_vote_diverse_no_retrieval` as the final submission candidate while reporting the macro-F1 caveat clearly.


## Figure 10: Model Comparison

What we expected: RoBERTa should beat classical baselines, but classical methods might still be competitive because literals are short and surface-form heavy.

What we found: RoBERTa models are clearly stronger than majority, retrieval, and TF-IDF baselines. The final ensemble improves beyond the best single RoBERTa model.

How it affects the next step: we choose `v10_vote_diverse_no_retrieval` as the final public candidate, while keeping `v09_ensemble` and the individual models for explanation and ablation evidence.

![Model comparison](../reports/figures/fig_10_model_comparison.png)


## Figure 11: Final Confusion Matrix

The confusion matrix shows where the final model sends examples when it is wrong. This is important because broad ICD prefix categories can be clinically related or share surface terms.

![Final confusion matrix](../reports/figures/fig_11_final_confusion_matrix.png)

## Figure 15: Top Confused Category Pairs

The confusion matrix is dense, so we also save the most frequent true-to-predicted error pairs as a bar chart. This makes recurring confusions such as `V -> Z` and `6 -> O` easier to discuss in the report.

![Top confused pairs](../reports/figures/fig_15_top_confused_pairs.png)


In [ ]:
confusions.head(15)


## Figure 12: Per-Class Recall

What we expected: rare categories should have lower recall because EDA showed imbalance and some labels have very few examples.

What we found: recall varies strongly by category. Some categories are learned well, while others remain weak. This is why macro F1 stays much lower than accuracy.

How it affects the next step: a deployed clinical tool would need uncertainty handling and probably more data or label-aware modeling for weak categories.

![Per-class recall](../reports/figures/fig_12_per_class_recall.png)


In [ ]:
per_class.sort_values('recall').head(12)


## Figure 13: Confidence Analysis

Confidence is useful, but it is not the same as correctness. We look at the model's top probability and the margin between the top two categories.

What we expected: correct predictions should usually have higher confidence and larger margins, but ambiguous literals can still produce high-confidence errors.

What we found: wrong predictions are more common at lower confidence, but there are also high-confidence mistakes. These are especially important for clinical interpretability.

![Confidence correct vs wrong](../reports/figures/fig_13_confidence_correct_vs_wrong.png)


## Real Literal Examples

We selected four groups from the final validation predictions:

- correct high-confidence;
- correct low-confidence;
- wrong high-confidence;
- wrong low-confidence.

These are not synthetic examples. They come from `outputs/predictions/v10_vote_diverse_no_retrieval_val_predictions.csv` and are saved in `reports/tables/final_error_examples.csv`.


In [ ]:
for group_name, group in errors.groupby('example_group'):
    display(group[['Literal', 'y_true', 'y_pred', 'confidence', 'confidence_margin', 'possible_error_reason']].head(5))


## Possible Causes of Errors

From the examples and confusion pairs, the main error causes are:

- **ambiguous literal**: the same or similar phrase may appear in different clinical contexts;
- **insufficient context**: many literals are only one or two words;
- **abbreviations**: uppercase abbreviations such as `IAM`, `TC`, or shorthand clinical forms may be hard to disambiguate;
- **class imbalance**: low-support categories receive fewer learning signals;
- **similar ICD categories**: several prefixes overlap in clinical practice or in how literals are written;
- **broad prefix categories**: predicting only the first ICD character simplifies full ICD coding, but it can also group heterogeneous cases.


## Figure 14: Training Curves

Training curves help us distinguish useful learning from overfitting. Several RoBERTa models improve quickly and then validation loss rises, which explains why early stopping and best-checkpoint selection were necessary.

![Training curves](../reports/figures/fig_14_training_curves.png)


## Interpretability and the ICD Survey

The ICD coding survey emphasizes interpretability because clinical coding affects statistics, reimbursement, hospital management, and medical-record organization. In our project, interpretability is limited but still useful:

- TF-IDF models can expose important character n-grams;
- confusion matrices show systematic category confusions;
- confidence and margin analysis show when the model is uncertain;
- representative examples help us explain failures in human terms.

We did **not** use attention/token attribution as a clinical explanation in this notebook. Attention weights or token attribution methods can be informative diagnostics, but they are not guaranteed faithful explanations of the model decision. If we add token attribution later, we should present it as exploratory, not as proof that the model reasons like a clinician.


## Final Candidate Decision

Based on validation, error analysis, and verified Kaggle public score, the current final candidate is:

```text
v10_vote_diverse_no_retrieval
validation accuracy: 0.579562
macro F1: 0.496677
weighted F1: 0.561294
Kaggle public score: 0.587
```

We trust it more than a single RoBERTa run because it improves validation accuracy, gives the best verified public score, and combines complementary signals from RoBERTa and TF-IDF. We still report its limitations clearly: errors remain on ambiguous short literals, rare categories, abbreviations, and broad ICD-prefix confusions.
